# DustyCult July 1 Dippers

Run DustyCult quick fits for the July 1 review candidates labeled `dipper` in `output/runs/dat3-full-extended_2026-07-01-v4/review/review.db`.

This notebook writes fit metadata and posterior predictive curves back into the review DB tables used by the review app:

- `dustycult_fits`
- `dustycult_predictive_curves`
- `output/runs/dat3-full-extended_2026-07-01-v4/review/dustycult/<candidate>/quick/`


## Setup

Run these first. The fitting cells near the end are the only cells that intentionally update the review DB and DustyCult artifact directories.

In [ ]:
from __future__ import annotations

import json
import math
import sqlite3
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

root_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        p for p in root_candidates
        if (p / 'pyproject.toml').exists()
        and (p / 'malca' / 'review' / 'dustycult.py').exists()
    ),
    Path.cwd(),
)

for path in (REPO_ROOT, REPO_ROOT / 'malca'):
    text = str(path.resolve())
    if text not in sys.path:
        sys.path.insert(0, text)

from malca.io.notebook_paths import resolve_local_lightcurve_path
from malca.review.dustycult import (
    check_dustycult_available,
    control_defaults_for_candidate,
    load_dustycult_curve,
    load_dustycult_fits,
    run_dustycult_fit,
)
from malca.review.dustycult_display import (
    build_dustycult_fit_figure,
    dustycult_fit_metadata_rows,
    dustycult_geometry_rows,
    dustycult_posterior_rows,
    select_dustycult_display_row,
)
from malca.review.dustycult_visualization import build_dustycult_occulter_figure
from malca.review.store import db_connect

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 160)
pd.set_option('display.width', 240)


## Configuration And Preflight

`MAX_CANDIDATES = 1` is the smoke-test default. The full-run cell below resets it to `None` and skips existing `ok` or `warning` quick fits.

In [ ]:
RUN_DIR = REPO_ROOT / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4'
DB_PATH = RUN_DIR / 'review' / 'review.db'
LIGHTCURVE_DIR = RUN_DIR / 'bundle_assets' / 'lightcurves'
PLOT_DIR = RUN_DIR / 'plots'
RESULTS_DIR = RUN_DIR / 'results'

MODES = ('quick',)
GOOD_STATUSES = {'ok', 'warning'}
RERUN_EXISTING = False
MAX_CANDIDATES = 1
EXPECTED_DIPPERS = 71
EXPECTED_MISSING_QUICK = 69
JULIA = 'julia'
DUSTYCULT_PROJECT = REPO_ROOT / 'external' / 'dustycult'
EXAMPLE_PLOT_MODE = 'quick'

RUN_PARAMS_PATH = RUN_DIR / 'run_params.json'
RUN_PARAMS = json.loads(RUN_PARAMS_PATH.read_text()) if RUN_PARAMS_PATH.exists() else {}

availability = check_dustycult_available(project_path=DUSTYCULT_PROJECT, julia=JULIA)

print(f'REPO_ROOT         = {REPO_ROOT}')
print(f'RUN_DIR           = {RUN_DIR}')
print(f'DB_PATH           = {DB_PATH}')
print(f'LIGHTCURVE_DIR    = {LIGHTCURVE_DIR}')
print(f'PLOT_DIR          = {PLOT_DIR}')
print(f'DUSTYCULT_PROJECT = {DUSTYCULT_PROJECT}')
print(f'DB exists         = {DB_PATH.exists()}')
print(f'LC dir exists     = {LIGHTCURVE_DIR.is_dir()}')
print(f'run_params exists = {RUN_PARAMS_PATH.exists()}')
print(f'Julia executable  = {availability.julia}')
print(f'DustyCult status  = {availability.message}')

if not DB_PATH.exists():
    raise FileNotFoundError(DB_PATH)
if not LIGHTCURVE_DIR.is_dir():
    raise FileNotFoundError(LIGHTCURVE_DIR)
if not availability.ok:
    raise RuntimeError(availability.message)


## Load July 1 Dipper Candidates

This uses the strict selector `reviews.event_class = 'dipper'`.

In [ ]:
DIPPER_WHERE = "r.event_class = 'dipper'"

with db_connect(DB_PATH) as conn:
    reviewed_dippers = pd.read_sql_query(
        f'''
        SELECT
            c.*,
            r.event_class,
            r.workflow_status,
            r.disposition,
            r.morphology_primary,
            r.morphology_secondary,
            r.morphology_secondary_json,
            r.physical_primary,
            r.physical_secondary,
            r.classification_confidence,
            r.priority_tags_json,
            r.notes AS review_notes,
            r.updated_at AS review_updated_at
        FROM reviews r
        JOIN candidates c USING(candidate_id)
        WHERE {DIPPER_WHERE}
        ORDER BY r.updated_at DESC, r.candidate_id
        ''',
        conn,
    )
    existing_fits = pd.read_sql_query(
        f'''
        SELECT candidate_id, mode, status, updated_at, runtime_sec, t0_jd, start_jd, end_jd,
               n_input_points, n_curve_points, artifact_dir, error
        FROM dustycult_fits
        WHERE candidate_id IN (
            SELECT r.candidate_id FROM reviews r WHERE {DIPPER_WHERE}
        )
        ORDER BY candidate_id, mode
        ''',
        conn,
    )

quick_fits = existing_fits[existing_fits['mode'].astype(str).eq('quick')].copy() if not existing_fits.empty else pd.DataFrame()
good_quick_ids = set(
    quick_fits.loc[quick_fits['status'].astype(str).isin(GOOD_STATUSES), 'candidate_id'].astype(str)
) if not quick_fits.empty else set()
missing_quick_count = int(len(reviewed_dippers) - len(good_quick_ids))

print(f'July 1 dippers: {len(reviewed_dippers)}')
print(f'Existing quick fits with ok/warning status: {len(good_quick_ids)}')
print(f'Missing quick fits by skip policy: {missing_quick_count}')

if len(reviewed_dippers) != EXPECTED_DIPPERS:
    display(Markdown(f'**Warning:** expected `{EXPECTED_DIPPERS}` dippers, found `{len(reviewed_dippers)}`.'))
if missing_quick_count != EXPECTED_MISSING_QUICK:
    display(Markdown(f'**Warning:** expected `{EXPECTED_MISSING_QUICK}` missing quick fits, found `{missing_quick_count}`.'))

display_cols = [
    'candidate_id', 'event_class', 'workflow_status', 'disposition',
    'morphology_primary', 'morphology_secondary', 'physical_primary',
    'dipper_score', 'dipper_n_valid_dips', 'dip_run_count', 'dip_best_morph',
]
display(reviewed_dippers[[col for col in display_cols if col in reviewed_dippers.columns]])
display(existing_fits if not existing_fits.empty else Markdown('No existing DustyCult fits for these dippers.'))


## Helpers

These mirror the review app path resolution and DustyCult fit calls, but keep execution serial for DB safety.

In [ ]:
def _finite_float(value, default=None):
    try:
        if value is None or pd.isna(value):
            return default
    except Exception:
        if value is None:
            return default
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default
    return number if math.isfinite(number) else default


def clean_value(value):
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


def row_payload(row: pd.Series) -> dict[str, object]:
    payload: dict[str, object] = {}
    raw = row.get('payload_json')
    if isinstance(raw, str) and raw.strip():
        try:
            payload.update(json.loads(raw))
        except Exception:
            pass
    for key, value in row.items():
        cleaned = clean_value(value)
        if cleaned is not None:
            payload[key] = cleaned
    payload['candidate_id'] = str(row['candidate_id'])
    return payload


def resolve_candidate_lc_path(row: pd.Series, payload: dict[str, object] | None = None) -> Path | None:
    payload = payload or row_payload(row)
    candidates = [
        row.get('lc_path'),
        payload.get('lc_path'),
        payload.get('path'),
        payload.get('dat_path'),
        payload.get('asas_sn_id'),
        row.get('candidate_id'),
    ]
    for value in candidates:
        if value is None:
            continue
        resolved = resolve_local_lightcurve_path(value, run_dir=RUN_DIR, repo_root=REPO_ROOT)
        if resolved is not None and resolved.exists():
            return resolved
    return None


def mode_status(fits: pd.DataFrame, candidate_id: str, mode: str) -> str | None:
    if fits.empty:
        return None
    sub = fits[(fits['candidate_id'].astype(str) == str(candidate_id)) & (fits['mode'].astype(str) == str(mode))]
    if sub.empty:
        return None
    if 'updated_at' in sub.columns:
        sub = sub.sort_values('updated_at')
    return str(sub.iloc[-1].get('status') or '') or None


def planned_mode_action(fits: pd.DataFrame, candidate_id: str, mode: str) -> str:
    status = mode_status(fits, candidate_id, mode)
    if RERUN_EXISTING:
        return 'rerun_existing'
    if status in GOOD_STATUSES:
        return 'skipped_existing_good'
    if status:
        return f'retry_existing_{status}'
    return 'run_missing'


def active_controls_for_candidate(
    conn: sqlite3.Connection,
    candidate_id: str,
    payload: dict[str, object],
    *,
    lc_path: Path | None,
) -> dict[str, object]:
    defaults = control_defaults_for_candidate(
        conn,
        candidate_id,
        payload,
        lc_path=lc_path,
        plot_dir=PLOT_DIR,
        run_params=RUN_PARAMS,
        recompute=False,
    )
    controls = {key: defaults.get(key) for key in defaults.keys()}
    controls['_dustycult_window_source'] = str(defaults.get('source') or 'defaults')
    return controls


def select_fit_candidates(max_candidates: int | None = None) -> pd.DataFrame:
    data = reviewed_dippers.copy()
    if max_candidates is not None:
        data = data.head(int(max_candidates)).copy()
    return data.reset_index(drop=True)


def reload_existing_fits(candidate_ids: list[str] | None = None) -> pd.DataFrame:
    ids = [str(item) for item in (candidate_ids or reviewed_dippers['candidate_id'].astype(str).tolist())]
    if not ids:
        return pd.DataFrame()
    placeholders = ','.join(['?'] * len(ids))
    with db_connect(DB_PATH) as conn:
        return pd.read_sql_query(
            f'''
            SELECT candidate_id, mode, status, updated_at, runtime_sec, t0_jd, start_jd, end_jd,
                   n_input_points, n_curve_points, artifact_dir, error
            FROM dustycult_fits
            WHERE candidate_id IN ({placeholders})
            ORDER BY candidate_id, mode
            ''',
            conn,
            params=ids,
        )


def build_dry_run_queue(max_candidates: int | None = None) -> pd.DataFrame:
    fit_candidates = select_fit_candidates(max_candidates)
    current_fits = reload_existing_fits(fit_candidates['candidate_id'].astype(str).tolist())
    rows = []
    with db_connect(DB_PATH) as conn:
        for _, row in fit_candidates.iterrows():
            candidate_id = str(row['candidate_id'])
            payload = row_payload(row)
            lc_path = resolve_candidate_lc_path(row, payload)
            controls = active_controls_for_candidate(conn, candidate_id, payload, lc_path=lc_path)
            quick_status = mode_status(current_fits, candidate_id, 'quick')
            rows.append(
                {
                    'candidate_id': candidate_id,
                    'lc_found': lc_path is not None,
                    'lc_path': str(lc_path) if lc_path else '',
                    'quick_status': quick_status or 'missing',
                    'quick_action': planned_mode_action(current_fits, candidate_id, 'quick'),
                    'window_source': controls.get('_dustycult_window_source'),
                    'start_jd': controls.get('start_jd'),
                    't0_jd': controls.get('t0_jd'),
                    'end_jd': controls.get('end_jd'),
                    'dipper_score': row.get('dipper_score'),
                    'dipper_n_valid_dips': row.get('dipper_n_valid_dips'),
                    'dip_run_count': row.get('dip_run_count'),
                    'dip_best_morph': row.get('dip_best_morph'),
                }
            )
    return pd.DataFrame(rows)


## Dry-Run Queue

This cell does not write anything. Use it before the smoke test and again before the full run.

In [ ]:
dry_run_queue = build_dry_run_queue(max_candidates=None)
display(dry_run_queue)
display(dry_run_queue.groupby(['quick_action', 'lc_found'], dropna=False).size().reset_index(name='n'))


## Run DustyCult Fits

These cells write to the review DB and create/recreate DustyCult artifacts for candidates that are not skipped.

In [ ]:
def record_result(results: list[dict[str, object]], candidate_id: str, mode: str, action: str, row: dict[str, object] | None = None, error: str = '') -> None:
    row = dict(row or {})
    results.append(
        {
            'candidate_id': str(candidate_id),
            'mode': mode,
            'action': action,
            'status': row.get('status', ''),
            'runtime_sec': row.get('runtime_sec'),
            't0_jd': row.get('t0_jd'),
            'start_jd': row.get('start_jd'),
            'end_jd': row.get('end_jd'),
            'n_input_points': row.get('n_input_points'),
            'n_curve_points': row.get('n_curve_points'),
            'artifact_dir': row.get('artifact_dir', ''),
            'error': row.get('error', error),
        }
    )


def run_fit_queue(max_candidates: int | None = None) -> pd.DataFrame:
    fit_candidates = select_fit_candidates(max_candidates)
    results: list[dict[str, object]] = []
    started = time.monotonic()

    with db_connect(DB_PATH) as conn:
        for idx, row in fit_candidates.iterrows():
            candidate_id = str(row['candidate_id'])
            payload = row_payload(row)
            lc_path = resolve_candidate_lc_path(row, payload)
            if lc_path is not None:
                payload['lc_path'] = str(lc_path)

            print(f'[{idx + 1}/{len(fit_candidates)}] {candidate_id}')
            fits_before = load_dustycult_fits(conn, candidate_id)
            controls = active_controls_for_candidate(conn, candidate_id, payload, lc_path=lc_path)

            quick_action = planned_mode_action(fits_before, candidate_id, 'quick')
            if quick_action == 'skipped_existing_good':
                status = mode_status(fits_before, candidate_id, 'quick') or 'skipped'
                record_result(results, candidate_id, 'quick', quick_action, {'status': status, **controls})
                print(f'  quick: skipped existing {status}')
                continue

            quick_row = run_dustycult_fit(
                conn,
                candidate_id,
                payload,
                db_path=DB_PATH,
                controls=controls,
                mode='quick',
                lc_path=lc_path,
                plot_dir=PLOT_DIR,
                run_params=RUN_PARAMS,
                project_path=DUSTYCULT_PROJECT,
                julia=JULIA,
            )
            record_result(results, candidate_id, 'quick', quick_action, quick_row)
            print(f"  quick: {quick_row.get('status')} {quick_action} {quick_row.get('error') or ''}")

    out = pd.DataFrame(results)
    elapsed = time.monotonic() - started
    print(f'Finished {len(out)} candidate-mode rows in {elapsed:.1f} s')
    return out


### Smoke Test

Run one candidate first. If it completes or records a sensible warning/failure, run the full queue cell below.

In [ ]:
MAX_CANDIDATES = 1
smoke_results = run_fit_queue(max_candidates=MAX_CANDIDATES)
display(smoke_results)

if not smoke_results.empty:
    smoke_id = str(smoke_results.iloc[0]['candidate_id'])
    with db_connect(DB_PATH) as conn:
        smoke_fits = load_dustycult_fits(conn, smoke_id)
    display(smoke_fits)
    artifact_dir = smoke_results.iloc[0].get('artifact_dir')
    if artifact_dir:
        artifact_path = Path(str(artifact_dir))
        print(f'Artifact dir exists: {artifact_path.exists()} -> {artifact_path}')


### Full Remaining Quick Run

This sets `MAX_CANDIDATES = None`. Existing `ok` or `warning` quick fits are skipped, including the smoke-test result if it succeeded or warned.

In [ ]:
MAX_CANDIDATES = None
full_results = run_fit_queue(max_candidates=MAX_CANDIDATES)
display(full_results)
display(full_results.groupby(['action', 'status'], dropna=False).size().reset_index(name='n'))


## Summary And Example Display

Use this after the smoke test or full run. It reloads from SQLite instead of trusting in-memory results.

In [ ]:
candidate_ids = reviewed_dippers['candidate_id'].astype(str).tolist()
final_fits = reload_existing_fits(candidate_ids)

display(final_fits.groupby(['mode', 'status'], dropna=False).size().reset_index(name='n') if not final_fits.empty else Markdown('No DustyCult fits found.'))
display(final_fits.sort_values(['candidate_id', 'mode']) if not final_fits.empty else final_fits)

quick_final = final_fits[final_fits['mode'].astype(str).eq('quick')].copy() if not final_fits.empty else pd.DataFrame()
quick_good = quick_final[quick_final['status'].astype(str).isin(GOOD_STATUSES)].copy() if not quick_final.empty else pd.DataFrame()
print(f'Quick ok/warning fits: {len(quick_good)} / {len(reviewed_dippers)}')


def _display_rows_table(rows: list[tuple[str, str]] | list[tuple[str, str, str, str]]) -> None:
    if not rows:
        return
    frame = pd.DataFrame(rows)
    display(frame)


def _load_selected_dustycult_fit(candidate_id: str, mode: str | None = None):
    with db_connect(DB_PATH) as conn:
        fits = load_dustycult_fits(conn, candidate_id)
        fit_row = select_dustycult_display_row(fits, mode=mode)
        if fit_row is None:
            raise ValueError(f'No DustyCult fit row found for {candidate_id}')
        selected_mode = str(fit_row.get('mode') or 'quick')
        curves = load_dustycult_curve(conn, candidate_id, selected_mode)
    return fit_row, curves


def display_dustycult_review_panel(candidate_id: str, mode: str | None = None) -> None:
    fit_row, curves = _load_selected_dustycult_fit(candidate_id, mode)
    selected_mode = str(fit_row.get('mode') or 'quick')
    display(Markdown(f'### `{candidate_id}` DustyCult `{selected_mode}`'))
    _display_rows_table(dustycult_fit_metadata_rows(fit_row))
    if curves is not None and not curves.empty:
        display(build_dustycult_fit_figure(curves, fit_row, theme='white'))
    try:
        display(build_dustycult_occulter_figure(fit_row, theme='white', grid_n=501))
    except Exception as exc:
        display(Markdown(f'Occulter plot unavailable: `{exc}`'))
    _display_rows_table(dustycult_geometry_rows(fit_row))
    _display_rows_table(dustycult_posterior_rows(fit_row, limit=None))


if quick_good.empty:
    display(Markdown('No ok/warning quick fits are available yet.'))
else:
    example = quick_good.sort_values(['status', 'candidate_id']).iloc[0]
    display_dustycult_review_panel(str(example['candidate_id']), EXAMPLE_PLOT_MODE)
